In [2]:
import pandas as pd
import sys

# Define the names of your two input Excel files here.
# Make sure these match the actual names of the files on your computer!
excel_file_1 = "AI_ModelingData.xlsx" 
excel_file_2 = "AI_Modelling2.xlsx"

# This is the final output file your Flask app uses
output_csv = "synthetic_plant_data.csv"

def convert_and_combine():
    print(f"Attempting to load: {excel_file_1} and {excel_file_2}...")
    
    try:
        # Read the Excel files into pandas DataFrames
        # (Requires 'openpyxl' to be installed: pip install openpyxl)
        df1 = pd.read_excel(excel_file_1)
        df2 = pd.read_excel(excel_file_2)
        
        print("Files loaded successfully. Combining data...")
        
        # Concatenate (stack) the two dataframes on top of each other
        # ignore_index=True resets the row numbers so they flow sequentially
        combined_df = pd.concat([df1, df2], ignore_index=True)
        
        # Export the combined dataframe to a single CSV file
        # index=False ensures we don't accidentally save the row numbers as a new column
        combined_df.to_csv(output_csv, index=False)
        
        print(f"✅ Success! Combined {len(combined_df)} total rows.")
        print(f"✅ Saved as: {output_csv}")
        
    except FileNotFoundError as e:
        print(f"\n❌ Error: Could not find one of the Excel files.")
        print(f"Details: {e}")
        print("Please ensure your Excel files are in the exact same folder as this script, and the names match the variables at the top of the script.")
    except Exception as e:
        print(f"\n❌ An unexpected error occurred: {e}")
        print("Did you remember to run 'pip install openpyxl' ?")

if __name__ == "__main__":
    convert_and_combine()

Attempting to load: AI_ModelingData.xlsx and AI_Modelling2.xlsx...
Files loaded successfully. Combining data...
✅ Success! Combined 73440 total rows.
✅ Saved as: synthetic_plant_data.csv


In [ ]:
from fastapi import FastAPI, HTTPException
from pydantic import BaseModel
import pandas as pd
from sklearn.ensemble import IsolationForest
from sklearn.preprocessing import LabelEncoder
import warnings

# Ignore minor background warnings
warnings.filterwarnings('ignore')

# --- 1. INITIALIZE THE API ---
app = FastAPI(
    title="Thermal Plant AI Diagnostic API",
    description="Send live sensor data to receive instant thermodynamic fault diagnoses.",
    version="1.0"
)

# --- 2. TRAIN THE AI ON STARTUP ---
# The AI trains itself the moment the API server boots up
print("Booting up API and training AI Brain...")
try:
    df = pd.read_csv('synthetic_plant_data.csv')
    
    label_encoder = LabelEncoder()
    df['sensor_encoded'] = label_encoder.fit_transform(df['maindes'])
    X = df[['value1', 'sensor_encoded']]
    
    doctor_model = IsolationForest(contamination=0.05, random_state=42)
    doctor_model.fit(X)
    print("AI Model successfully trained and ready for API requests!")
except Exception as e:
    doctor_model = None
    label_encoder = None
    print(f"WARNING: Could not train model. {e}")

# --- 3. DEFINE THE DATA FORMAT ---
# This tells the API exactly what shape of data to expect from outside machines
class SensorReadings(BaseModel):
    value1: float
    maindes: str

# --- 4. CREATE THE ENDPOINTS ---

# Endpoint A: The Health Check
@app.get("/")
def read_root():
    return {"status": "online", "message": "Thermal Plant AI API is running."}

# Endpoint B: The Live Diagnosis (Replaces Tab 1 of the dashboard)
@app.post("/diagnose")
def diagnose_live_system(data: SensorReadings):
    if doctor_model is None:
        raise HTTPException(status_code=500, detail="AI Model is offline. Training data missing.")
    
    try:
        encoded_sensor = label_encoder.transform([data.maindes])[0]
    except:
        encoded_sensor = 0
        
    # 1. Convert the incoming API JSON data into a pandas format the AI understands
    live_sensors = pd.DataFrame([[data.value1, encoded_sensor]], columns=['value1', 'sensor_encoded'])
    
    # 2. Get the AI's prediction
    prediction = int(doctor_model.predict(live_sensors)[0])
    
    # 3. Translate the code
    if prediction == -1:
        diagnosis_text = f"Anomaly Detected in {data.maindes}"
    else:
        diagnosis_text = "System Normal"
    
    # 4. Return the diagnosis back to the machine that asked for it
    return {
        "timestamp": pd.Timestamp.now().isoformat(),
        "fault_code": prediction,
        "diagnosis": diagnosis_text,
        "confidence_check": "Complete"
    }